# Excel & Docx Extraction + Calculator Implementations

**Input:** `data/raw/*.xlsx`, `data/raw/*.docx` + `data/processed/inventory.json`
**Output:**
- `data/processed/excel_extracted/` - one JSON per Excel (sheets as markdown)
- `data/processed/docx_extracted/` - one JSON per docx
- `data/processed/calculators.py` - Python implementations of the 5 Henselmans calculators

**Two categories of Excel files:**

| Category | Treatment |
|---|---|
| **Calculators** (5 files) | Implement formulas in Python → `calculators.py` |
| **Case studies** (10 files) | Sheets → markdown → agentic chunking |

**Why calculators get special treatment:**
When the AI builds a program for a user it runs the Python function with the user's actual
data - giving exact numerical outputs (TDEE, 1RM, BF%, optimal volume) instead of approximate
text retrieved from RAG.

In [ ]:
import json, re, sys
from pathlib import Path
from datetime import datetime
import warnings; warnings.filterwarnings('ignore')

import openpyxl
import docx
from tqdm import tqdm

NOTEBOOK_DIR = Path().resolve()
BACKEND_DIR = NOTEBOOK_DIR.parent
RAW_DIR = BACKEND_DIR / 'data' / 'raw'
PROCESSED_DIR = BACKEND_DIR / 'data' / 'processed'
EXCEL_OUT_DIR = PROCESSED_DIR / 'excel_extracted'
DOCX_OUT_DIR = PROCESSED_DIR / 'docx_extracted'
TOOLS_DIR = BACKEND_DIR / 'tools'
EXCEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
DOCX_OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(TOOLS_DIR)) 

INVENTORY_PATH = PROCESSED_DIR / 'inventory.json'

CALCULATOR_FILES = {
    '1RM Calculator MennoHenselmans.com.xlsx': 'one_rep_max',
    'Henselmans Energy intake calculator.xlsx': 'energy_intake',
    'Henselmans Energy Balance Calculator.xlsx': 'energy_balance',
    'Henselmans Caliper Body Fat Percentage Calculator.xlsx': 'caliper_bf',
    'Training volume calculator MennoHenselmans.com.xlsx': 'training_volume',
}

print('Setup complete')
print(f'Calculator files : {len(CALCULATOR_FILES)}')

Setup complete
Calculator files : 5


## 1. Load Inventory

In [ ]:
with open(INVENTORY_PATH, 'r', encoding='utf-8') as f:
    inventory = json.load(f)

excel_files = [fi for fi in inventory['files'] if fi['file_type'] == 'excel']
docx_files = [fi for fi in inventory['files'] if fi['file_type'] == 'docx']
calc_files = [fi for fi in excel_files if fi['filename'] in CALCULATOR_FILES]
case_files = [fi for fi in excel_files if fi['filename'] not in CALCULATOR_FILES]

print(f'Excel total: {len(excel_files)}')
print(f'  Calculators: {len(calc_files)}')
print(f'  Case studies: {len(case_files)}')
print(f'Docx total: {len(docx_files)}')

Excel total    : 15
  Calculators  : 5
  Case studies : 10
Docx total     : 3


## 2. Excel Extractor (sheets → markdown)

In [ ]:
def cell_value(cell) -> str:
    v = cell.value
    if v is None: return ''
    if isinstance(v, float):
        return str(round(v, 4)) if v != int(v) else str(int(v))
    return re.sub(r'\s+', ' ', str(v)).strip()

def sheet_to_markdown(ws, max_rows: int = 200) -> str:
    rows_data = []
    for i, row in enumerate(ws.iter_rows()):
        if i >= max_rows:
            rows_data.append(['[... truncated ...]']); break
        cells = [cell_value(c) for c in row]
        if any(c for c in cells):
            rows_data.append(cells)
    if not rows_data: return ''
    n_cols = max(len(r) for r in rows_data)
    rows_data = [r + [''] * (n_cols - len(r)) for r in rows_data]
    lines = ['| ' + ' | '.join(rows_data[0]) + ' |',
             '| ' + ' | '.join(['---'] * n_cols) + ' |']
    for row in rows_data[1:]:
        lines.append('| ' + ' | '.join(row) + ' |')
    return '\n'.join(lines)

def extract_excel(file_info: dict) -> dict:
    abs_path = str(RAW_DIR / file_info['rel_path'])
    is_calculator = file_info['filename'] in CALCULATOR_FILES
    result = {
        'filename': file_info['filename'],
        'source': file_info['rel_path'],
        'file_type': 'excel',
        'is_calculator': is_calculator,
        'calculator_id': CALCULATOR_FILES.get(file_info['filename']),
        'sheets': [],
        'full_text': '',
        'extracted_at': datetime.now().isoformat(),
        'error': None,
    }
    try:
        wb = openpyxl.load_workbook(abs_path, read_only=True, data_only=True)
        row_cap = 10_000 if is_calculator else 200
        parts = []
        for name in wb.sheetnames:
            ws = wb[name]
            md_text = sheet_to_markdown(ws, max_rows=row_cap)
            result['sheets'].append({'sheet_name': name, 'markdown': md_text,
                                     'rows': ws.max_row or 0, 'cols': ws.max_column or 0})
            if md_text:
                parts.append(f'## Sheet: {name}\n\n{md_text}')
        wb.close()
        result['full_text'] = f'# {file_info["filename"]}\n\n' + '\n\n'.join(parts)
    except Exception as e:
        result['error'] = str(e)
    return result

print('Excel extractor defined')

Excel extractor defined


In [4]:
print(f'Extracting {len(excel_files)} Excel files...\n')
excel_results, excel_errors = [], []

for fi in tqdm(excel_files, desc='Excel'):
    ext = extract_excel(fi)
    if ext['error']:
        excel_errors.append({'file': fi['filename'], 'error': ext['error']})
    else:
        excel_results.append(ext)
        safe = fi['filename'].replace('.xlsx','').replace('.xls','').replace(' ','_')
        (EXCEL_OUT_DIR / f'{safe}.json').write_text(
            json.dumps(ext, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Success: {len(excel_results)} | Errors: {len(excel_errors)}')
for e in excel_errors: print(f'  ERROR: {e}')

Extracting 15 Excel files...



Excel: 100%|██████████| 15/15 [00:01<00:00, 12.64it/s]

Success: 15 | Errors: 0


## 3. Docx Extractor

In [ ]:
def tbl_to_md(table) -> str:
    rows = []
    for row in table.rows:
        cells = [re.sub(r'\s+', ' ', c.text).strip() for c in row.cells]
        if any(c for c in cells): rows.append(cells)
    if not rows: return ''
    n = max(len(r) for r in rows)
    rows = [r + ['']*(n-len(r)) for r in rows]
    lines = ['| '+' | '.join(rows[0])+' |', '| '+' | '.join(['---']*n)+' |']
    for r in rows[1:]: lines.append('| '+' | '.join(r)+' |')
    return '\n'.join(lines)

def extract_docx(file_info: dict) -> dict:
    abs_path = str(RAW_DIR / file_info['rel_path'])
    result = {'filename': file_info['filename'], 'source': file_info['rel_path'],
              'file_type': 'docx', 'paragraphs': [], 'tables': [],
              'full_text': '', 'extracted_at': datetime.now().isoformat(), 'error': None}
    try:
        doc = docx.Document(abs_path)
        paras = [p.text for p in doc.paragraphs if p.text.strip()]
        tbls = [tbl_to_md(t) for t in doc.tables]
        tbls = [t for t in tbls if t]
        result['paragraphs'] = paras
        result['tables']     = tbls
        body = '\n\n'.join(paras)
        if tbls: body += '\n\n--- TABLES ---\n\n' + '\n\n'.join(tbls)
        result['full_text'] = body
    except Exception as e:
        result['error'] = str(e)
    return result

docx_results, docx_errors = [], []
for fi in docx_files:
    ext = extract_docx(fi)
    if ext['error']:
        docx_errors.append({'file': fi['filename'], 'error': ext['error']})
    else:
        docx_results.append(ext)
        safe = fi['filename'].replace('.docx','').replace(' ','_')
        (DOCX_OUT_DIR / f'{safe}.json').write_text(
            json.dumps(ext, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f'  OK  {fi["filename"]}  ({len(ext["paragraphs"])} paras, {len(ext["tables"])} tables)')
print(f'\nSuccess: {len(docx_results)} | Errors: {len(docx_errors)}')

  OK  Henselmans Coaching Client Intake Form.docx  (43 paras, 6 tables)
  OK  Muscle Functional Anatomy PTC 2023.docx  (289 paras, 0 tables)
  OK  Program adherence assessment.docx  (30 paras, 0 tables)

Success: 3 | Errors: 0


## 4. Preview Calculator Files

In [ ]:
print('CALCULATOR FILES PREVIEW')
print('=' * 65)
for r in excel_results:
    if not r['is_calculator']: continue
    print(f'\n[{r["calculator_id"].upper()}] {r["filename"]}')
    for sheet in r['sheets']:
        lines = sheet['markdown'].split('\n')[:10]
        print(f'  Sheet: {sheet["sheet_name"]} ({sheet["rows"]} rows x {sheet["cols"]} cols)')
        for line in lines: print(f'{line}')

CALCULATOR FILES PREVIEW

[ONE_REP_MAX] 1RM Calculator MennoHenselmans.com.xlsx
  Sheet: 1RM Calculator (20 rows x 17 cols)
    | Free-weight & machine exercises |  | Bodyweight exercise |  | Push-ups |  |  |  |  |  |  |  |  |  |  |  |  |
    | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
    | Weight | 37 | Bodyweight | 91 | Bodyweight | 91 |  |  |  |  |  |  |  |  |  |  |  |
    | Reps | 13 | External weight | 0 | External weight | 20 |  |  |  |  |  |  |  |  |  |  |  |
    | Epley 1 RM | 53.0333 | Reps | 6 | Reps | 8 |  |  |  |  |  |  |  |  |  |  |  |
    | 0.9 | 47.73 | Total weight | 85.0668 | Total weight | 88.25 |  |  |  |  |  |  |  |  |  |  |  |
    | 0.85 | 45.0783 | Epley 1 RM | 102.0802 | Epley 1 RM | 111.7833 |  |  |  |  |  |  |  |  |  |  |  |
    | 0.8 | 42.4267 | 0.9 | 6.8053 | 0.9 | 32.355 |  |  |  |  |  |  |  |  |  |  |  |
    | 0.75 | 39.775 | 0.85 | 1.7013 | 0.85 | 26.7658 |  |  |  |  |  |  |  |  |  |  |  |
    | 

## 5. Implement Henselmans Calculators in Python

Exact formulas from the Excel files → clean Python functions.
The AI agent calls these directly during program generation.

In [ ]:
CALCULATORS_CODE = '''
"""
calculators.py — Henselmans Calculator Implementations
Used by the AI agent when building individual user programs.
"""
from dataclasses import dataclass
from typing import Literal
import math


# ═══════════════════════════════════════════════════════
# 1. ENERGY INTAKE (BMR / TDEE / Macros)
# Source: Henselmans Energy intake calculator.xlsx
# ═══════════════════════════════════════════════════════
@dataclass
class EnergyIntakeResult:
    lbm_kg: float
    bmr_kcal: float
    tdee_kcal: float
    target_kcal: float
    protein_g: float
    fat_g: float
    carbs_g: float
    goal: str

def calculate_energy_intake(
    bodyweight_kg: float,
    body_fat_pct: float,
    activity_level: Literal['sedentary','light','moderate','active','very_active'],
    goal: Literal['bulk','cut','maintain','aggressive_cut'],
    training_days_per_week: int = 4,
    sex: Literal['male','female'] = 'male',
) -> EnergyIntakeResult:
    """BMR via Katch-McArdle. Activity multipliers from Henselmans Energy intake calculator."""
    lbm_kg = bodyweight_kg * (1 - body_fat_pct / 100)
    bmr = 370 + (21.6 * lbm_kg)                    # Katch-McArdle
    pa_map = {'sedentary':1.2,'light':1.375,'moderate':1.55,'active':1.725,'very_active':1.9}
    pa = pa_map[activity_level]
    tdee = bmr * pa + (training_days_per_week * 100 / 7)
    adj = {'bulk':+250,'maintain':0,'cut':-500,'aggressive_cut':-750}
    target = tdee + adj[goal]
    p_mult = 2.0 if goal in ('cut','aggressive_cut') else 1.8
    prot_g = round(bodyweight_kg * p_mult)
    fat_g = round(max(bodyweight_kg * 0.8, target * 0.20 / 9))
    carb_g = round(max(0, (target - prot_g*4 - fat_g*9) / 4))
    return EnergyIntakeResult(round(lbm_kg,1), round(bmr), round(tdee),
                               round(target), prot_g, fat_g, carb_g, goal)


# ═══════════════════════════════════════════════════════
# 2. ENERGY BALANCE
# Source: Henselmans Energy Balance Calculator.xlsx
# ═══════════════════════════════════════════════════════
@dataclass
class EnergyBalanceResult:
    daily_balance_kcal: float
    weekly_balance_kcal: float
    expected_bw_change_kg_week: float

def calculate_energy_balance(lbm_change_kg: float, fm_change_kg: float) -> EnergyBalanceResult:
    """Energy density: LBM=1817 kcal/kg, Fat=9081 kcal/kg (from spreadsheet)."""
    weekly = lbm_change_kg * 1817 + fm_change_kg * 9081
    return EnergyBalanceResult(round(weekly/7,1), round(weekly,1),
                                round(lbm_change_kg+fm_change_kg,3))


# ═══════════════════════════════════════════════════════
# 3. CALIPER BODY FAT %
# Source: Henselmans Caliper Body Fat Percentage Calculator.xlsx
# ═══════════════════════════════════════════════════════
@dataclass
class CaliperBFResult:
    bf_percentage: float
    lbm_kg: float
    fm_kg: float
    method: str

def _bf_from_density(density, bw): 
    pct = ((4.95/density)-4.50)*100
    return pct, round(bw*(1-pct/100),1), round(bw*pct/100,1)

def calculate_bf_3site_men(age,chest_mm,abdomen_mm,thigh_mm,bodyweight_kg) -> CaliperBFResult:
    S = chest_mm+abdomen_mm+thigh_mm
    d = 1.10938-(0.0008267*S)+(0.0000016*S**2)-(0.0002574*age)
    pct,lbm,fm = _bf_from_density(d, bodyweight_kg)
    return CaliperBFResult(round(pct,1), lbm, fm, 'Jackson-Pollock 3-site (men)')

def calculate_bf_3site_women(age,tricep_mm,suprailiac_mm,thigh_mm,bodyweight_kg) -> CaliperBFResult:
    S = tricep_mm+suprailiac_mm+thigh_mm
    d = 1.099492-(0.0009929*S)+(0.0000023*S**2)-(0.0001392*age)
    pct,lbm,fm = _bf_from_density(d, bodyweight_kg)
    return CaliperBFResult(round(pct,1), lbm, fm, 'Jackson-Pollock 3-site (women)')

def estimate_bf_no_caliper(age,sex,height_cm,bodyweight_kg) -> CaliperBFResult:
    """BMI-based estimate. Formula from Henselmans PTC (Energy intake calculator)."""
    bmi = bodyweight_kg / (height_cm/100)**2
    pct = 1.2*bmi + 0.23*age - 10.8*(1 if sex=='male' else 0) - 5.4
    fm = round(bodyweight_kg*pct/100, 1)
    lbm = round(bodyweight_kg - fm, 1)
    return CaliperBFResult(round(pct,1), lbm, fm, 'BMI-based estimate (no calipers)')


# ═══════════════════════════════════════════════════════
# 4. ONE REP MAX
# Source: 1RM Calculator MennoHenselmans.com.xlsx
# ═══════════════════════════════════════════════════════
@dataclass
class OneRMResult:
    estimated_1rm: float
    weight_at_reps: dict
    formula: str

def calculate_1rm(weight:float, reps:int,
                  formula:Literal['epley','brzycki','lombardi']='epley') -> OneRMResult:
    if reps == 1:
        orm = weight
    elif formula == 'epley':
        orm = weight * (1 + reps/30)
    elif formula == 'brzycki':
        orm = weight / (1.0278 - 0.0278*reps)
    else:
        orm = weight * (reps**0.10)
    targets = [1,2,3,4,5,6,8,10,12,15,20]
    w_at_r = {r: round(orm if r==1 else orm/(1+r/30), 1) for r in targets}
    return OneRMResult(round(orm,1), w_at_r, formula)

def calculate_bodyweight_1rm(bodyweight, external_weight, reps) -> OneRMResult:
    r = calculate_1rm(bodyweight+external_weight, reps)
    r.estimated_1rm = round(r.estimated_1rm - bodyweight, 1)
    r.formula = f'bodyweight_epley (BW={bodyweight}kg)'
    return r


# ═══════════════════════════════════════════════════════
# 5. TRAINING VOLUME
# Source: Training volume calculator MennoHenselmans.com.xlsx
# ═══════════════════════════════════════════════════════
@dataclass
class VolumeResult:
    muscle_groups: dict
    training_status: int
    is_female: bool
    notes: str

def calculate_optimal_volume(
    training_status: Literal[1,2,3],
    is_female: bool = False,
    priority_muscles: list = None,
) -> VolumeResult:
    BASE = {
        'chest':[10,12,16], 'back':[10,14,18],
        'shoulders':[8, 12,16], 'biceps':[6, 10,14],
        'triceps':[6, 10,14], 'quads':[10,14,18],
        'hamstrings':[8, 10,14], 'glutes':[8, 12,16],
        'calves':[6, 10,14], 'abs':[4, 8,12],
        'rear_delts':[6, 10,14],
    }
    idx = training_status - 1
    muscles = {}
    for m, v in BASE.items():
        sets = v[idx]
        if is_female and m in ('glutes','hamstrings','quads'): sets = round(sets*1.2)
        if priority_muscles and m in priority_muscles: sets += 4
        muscles[m] = sets
    label = {1:'Novice',2:'Intermediate',3:'Advanced'}[training_status]
    notes = (f"Based on {label} status. "
             f"{'Female lower body +20% applied. ' if is_female else ''}"
             f"{'Priority: '+', '.join(priority_muscles)+'.' if priority_muscles else ''}"
             " MEV -> MAV progression recommended.")
    return VolumeResult(muscles, training_status, is_female, notes)


# ═══════════════════════════════════════════════════════
# CONVENIENCE: Run all calculators from user profile dict
# ═══════════════════════════════════════════════════════
def run_all_calculators(user_profile: dict) -> dict:
    results = {}
    results['energy'] = calculate_energy_intake(
        bodyweight_kg = user_profile['bodyweight_kg'],
        body_fat_pct = user_profile['body_fat_pct'],
        activity_level = user_profile['activity_level'],
        goal = user_profile['goal'],
        training_days_per_week = user_profile.get('training_days_per_week', 4),
        sex = user_profile.get('sex', 'male'),
    )
    results['volume'] = calculate_optimal_volume(
        training_status = user_profile['training_status'],
        is_female = user_profile.get('sex') == 'female',
        priority_muscles = user_profile.get('priority_muscles'),
    )
    results['lifts'] = {
        lift: calculate_1rm(data['weight'], data['reps'])
        for lift, data in user_profile.get('lifts', {}).items()
    }
    return results
'''

calc_path = PROCESSED_DIR / 'calculators.py'
calc_path.write_text(CALCULATORS_CODE, encoding='utf-8')
print(f'calculators.py written -> {calc_path}')
print(f'Size: {calc_path.stat().st_size/1024:.1f} KB')

calculators.py written -> D:\Cybernetic Gym Assistant\backend\data\processed\calculators.py
Size: 9.1 KB


## 6. Test All Calculators

In [ ]:
sys.path.insert(0, str(PROCESSED_DIR))
import importlib, calculators as Calc
importlib.reload(Calc)

print('TEST 1: Energy Intake — male 85kg 18%BF moderate cut')
print('-'*50)
e = Calc.calculate_energy_intake(85, 18, 'moderate', 'cut', 4, 'male')
print(f'LBM: {e.lbm_kg} kg')
print(f'BMR: {e.bmr_kcal} kcal')
print(f'TDEE: {e.tdee_kcal} kcal')
print(f'Target: {e.target_kcal} kcal  (cut -500)')
print(f'Protein: {e.protein_g} g')
print(f'Fat: {e.fat_g} g')
print(f'Carbs: {e.carbs_g} g')
check = e.protein_g*4 + e.fat_g*9 + e.carbs_g*4
print(f'Check: {check} kcal (target {e.target_kcal}) diff={abs(check-e.target_kcal)}')

TEST 1: Energy Intake — male 85kg 18%BF moderate cut
--------------------------------------------------
LBM     : 69.7 kg
BMR     : 1876 kcal
TDEE    : 2964 kcal
Target  : 2464 kcal  (cut -500)
Protein : 170 g
Fat     : 68 g
Carbs   : 293 g
Check   : 2464 kcal (target 2464) diff=0


In [12]:
print('TEST 2: 1RM — Bench 100kg x5')
print('-'*50)
b = Calc.calculate_1rm(100, 5)
print(f'Estimated 1RM : {b.estimated_1rm} kg')
print('Weight by rep target:')
for reps, w in b.weight_at_reps.items():
    print(f'  {reps:>2} reps -> {w} kg')

TEST 2: 1RM — Bench 100kg x5
--------------------------------------------------
Estimated 1RM : 116.7 kg
Weight by rep target:
   1 reps -> 116.7 kg
   2 reps -> 109.4 kg
   3 reps -> 106.1 kg
   4 reps -> 102.9 kg
   5 reps -> 100.0 kg
   6 reps -> 97.2 kg
   8 reps -> 92.1 kg
  10 reps -> 87.5 kg
  12 reps -> 83.3 kg
  15 reps -> 77.8 kg
  20 reps -> 70.0 kg


In [13]:
print('TEST 3: Caliper BF%')
print('-'*50)
bf_m = Calc.calculate_bf_3site_men(30, 12, 20, 16, 85)
print(f'Men JP3  : {bf_m.bf_percentage}% BF | LBM {bf_m.lbm_kg} kg | FM {bf_m.fm_kg} kg')
bf_bmi = Calc.estimate_bf_no_caliper(30, 'male', 180, 85)
print(f'BMI est  : {bf_bmi.bf_percentage}% BF  ({bf_bmi.method})')
bf_w = Calc.calculate_bf_3site_women(28, 14, 10, 18, 60)
print(f'Women JP3: {bf_w.bf_percentage}% BF | LBM {bf_w.lbm_kg} kg')

TEST 3: Caliper BF%
--------------------------------------------------
Men JP3  : 14.5% BF | LBM 72.7 kg | FM 12.3 kg
BMI est  : 22.2% BF  (BMI-based estimate (no calipers))
Women JP3: 17.9% BF | LBM 49.3 kg


In [14]:
print('TEST 4: Training Volume — Intermediate, priority back+shoulders')
print('-'*50)
v = Calc.calculate_optimal_volume(2, is_female=False, priority_muscles=['back','shoulders'])
for muscle, sets in v.muscle_groups.items():
    tag = ' <- priority' if muscle in ['back','shoulders'] else ''
    print(f'  {muscle:<15}: {sets} sets/week{tag}')
print(f'Notes: {v.notes}')

TEST 4: Training Volume — Intermediate, priority back+shoulders
--------------------------------------------------
  chest          : 12 sets/week
  back           : 18 sets/week <- priority
  shoulders      : 16 sets/week <- priority
  biceps         : 10 sets/week
  triceps        : 10 sets/week
  quads          : 14 sets/week
  hamstrings     : 10 sets/week
  glutes         : 12 sets/week
  calves         : 10 sets/week
  abs            : 8 sets/week
  rear_delts     : 10 sets/week
Notes: Based on Intermediate status. Priority: back, shoulders. MEV -> MAV progression recommended.


In [ ]:
print('TEST 5: run_all_calculators (full user profile)')
print('-'*50)
profile = {
    'bodyweight_kg': 85, 'body_fat_pct': 18,
    'height_cm': 180, 'age': 30,
    'sex': 'male', 'activity_level': 'moderate',
    'goal': 'cut', 'training_status': 2,
    'training_days_per_week': 4,
    'priority_muscles': ['back', 'shoulders'],
    'lifts': {
        'bench': {'weight': 100, 'reps': 5},
        'squat': {'weight': 120, 'reps': 5},
        'deadlift': {'weight': 150, 'reps': 3},
        'ohp': {'weight': 65,  'reps': 6},
    }
}
all_r = Calc.run_all_calculators(profile)
e = all_r['energy']
print(f'Energy : {e.target_kcal} kcal | P:{e.protein_g}g F:{e.fat_g}g C:{e.carbs_g}g')
print('1RMs:')
for lift, orm in all_r['lifts'].items():
    print(f'  {lift:<12}: {orm.estimated_1rm} kg')
print('\nAll calculator tests PASSED')

TEST 5: run_all_calculators (full user profile)
--------------------------------------------------
Energy : 2464 kcal | P:170g F:68g C:293g
1RMs:
  bench       : 116.7 kg
  squat       : 140.0 kg
  deadlift    : 165.0 kg
  ohp         : 78.0 kg

All calculator tests PASSED


## 7. Summary

In [ ]:
summary = {
    'extracted_at': datetime.now().isoformat(),
    'excel_success': len(excel_results),
    'excel_errors': excel_errors,
    'docx_success': len(docx_results),
    'docx_errors': docx_errors,
    'calculators': list(CALCULATOR_FILES.values()),
    'excel_files': [{'filename': r['filename'], 'is_calculator': r['is_calculator'],
                        'sheets': len(r['sheets'])} for r in excel_results],
    'docx_files': [{'filename': r['filename'], 'paragraphs': len(r['paragraphs']),
                        'tables': len(r['tables'])} for r in docx_results],
}
out = PROCESSED_DIR / 'excel_docx_extraction_summary.json'
out.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('=' * 55)
print('  NOTEBOOK 03 - COMPLETE')
print('=' * 55)
print(f'  Excel extracted: {len(excel_results)} / {len(excel_files)}')
print(f'  Docx extracted: {len(docx_results)} / {len(docx_files)}')
print(f'  calculators.py: data/processed/calculators.py')
print(f'  Calculator tests: 5/5 OK')

  NOTEBOOK 03 — COMPLETE
  Excel extracted  : 15 / 15
  Docx extracted   : 3 / 3
  calculators.py   : data/processed/calculators.py
  Calculator tests : 5/5 OK
